# Propensão de compra de seguro — pipeline de lead scoring (telemarketing ativo)

**Objetivo:** dada uma base de clientes, ordenar quem tem maior probabilidade de comprar o seguro, para que a operação de discagem ligue primeiro para os leads mais promissores.

**Desafio central:** a base é enorme e a conversão é rara. No seu caso, ~8,88 milhões de registros e ~20.785 vendas, ou seja, **cerca de 0,23% de positivos**. Isso é um problema de *classificação extremamente desbalanceada*, e quase toda decisão do pipeline gira em torno disso.

Este notebook foi escrito para você **aprender o raciocínio**, não só rodar código. Cada seção começa com um bloco em markdown explicando *por que* estamos fazendo aquilo.

---

### Roteiro
1. Contexto e definição do problema
2. Dados sintéticos (com o mesmo schema — `df_raw`) e como plugar a base real
3. EDA e diagnóstico de desbalanceamento
4. Vazamento de dados (*data leakage*) — o erro mais perigoso aqui
5. Feature engineering e pré-processamento
6. Split temporal (por que **não** usar split aleatório)
7. Modelo baseline — Regressão Logística
8. Modelo foco — Gradient Boosting
9. Métricas certas para o problema (PR-AUC, precision@k, lift)
10. Escolha do ponto de operação (quantos ligar?)
11. Explicabilidade
12. **A pergunta causal do hotlist** — eles convertem porque são hotlist, ou porque o operador trabalha mais neles?
13. Conclusões e próximos passos


## 1. Contexto e definição do problema

Estamos em **telemarketing ativo**: a empresa liga para o cliente e oferece o produto. O modelo não decide *se* vamos vender — ele decide **a ordem de discagem**. A operação tem capacidade limitada (número de operadores × horas), então na prática ligamos para os *top-k* clientes de uma lista ordenada pela probabilidade prevista.

Isso muda tudo. Não interessa acertar a probabilidade absoluta de cada cliente; interessa que, **quando ordenamos a base pelo score do modelo, os compradores fiquem no topo**. Por isso as métricas de ranking (PR-AUC, precision@k, lift) importam muito mais do que "acurácia".

Por que **não** usamos acurácia? Com 0,23% de positivos, um modelo que prevê "ninguém compra" acerta 99,77% dos casos e é completamente inútil. A acurácia é enganosa em bases desbalanceadas; vamos ignorá-la.

**O conceito de abordagem "foco"** (que você escolheu): em vez de treinar 6 modelos de uma vez, começamos com um *baseline* simples e interpretável — a Regressão Logística — que serve de régua. Depois treinamos **um** modelo mais forte, *gradient boosting*, que costuma ser o melhor custo-benefício em dados tabulares. Comparamos os dois com as mesmas métricas. Você aprende o raciocínio sem o ruído de muitos modelos, e a estrutura já fica pronta para plugar mais modelos depois (deixamos indicado onde).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve)
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (7, 4)
print("Ambiente pronto. sklearn importado com sucesso.")

## 2. Dados sintéticos com o mesmo schema

Como a base real não pode ser compartilhada (compliance), este notebook gera uma **base sintética** que imita o seu schema: **as mesmas colunas, com os mesmos nomes**, o mesmo desbalanceamento (~0,23%) e algum sinal aprendível. Assim o notebook roda de ponta a ponta e você valida a lógica.

> **Importante:** o resultado é o DataFrame `df_raw` com os nomes de coluna idênticos aos seus. Para usar a base real, **basta substituir a célula de geração pela célula de carga** logo abaixo (comentada) — o resto do notebook não muda.

O gerador embute de propósito uma armadilha realista: **`fl_hl` (hotlist) está correlacionado com a conversão**, o que vamos investigar criticamente na Seção 12.


In [ ]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def gerar_base_sintetica(n=300_000, taxa_alvo=0.0023, random_state=RANDOM_STATE):
    """Gera df_raw com o mesmo schema da base real (nomes de coluna idênticos).

    n           : nº de registros (a base real tem ~8,88M; usamos menos p/ rodar rápido).
    taxa_alvo   : proporção de positivos desejada (~0,23%).
    """
    rng = np.random.default_rng(random_state)

    ufs = ["SP","RJ","MG","RS","PR","BA","SC","PE","CE","GO","DF","ES","PA","MA","PB"]
    bandeiras = ["VISA","MASTERCARD","ELO","AMEX"]
    tipos = ["STANDARD","GOLD","PLATINUM","SIGNATURE","INFINITE"]
    personas = ["FAMILIA_CONSERVADORA","CONECTADO","JOVEM_URBANO",
                "PREMIUM_VIAJANTE", "PRATICO", "INVESTIDOR"]
    prefs = ["RESTAURANTES","VIAGENS","SUPERMERCADO","COMBUSTIVEL",
             "SAUDE","EDUCACAO","VESTUARIO","STREAMING"]
    produtos = [(101,"SEGURO_PROTECAO_PESSOAL"),(102,"SEGURO_RESIDENCIAL"),
                (103,"SEGURO_HOSPITALAR")]

    uf = rng.choice(ufs, size=n)
    # idade e apenas VARIAVEL INTERNA para gerar dt_nascimento e o sinal;
    # ela NAO entra em df_raw (a base real so traz dt_nascimento).
    idade = np.clip(rng.normal(45, 14, n), 18, 90).astype(int)
    tipo_cartao = rng.choice(tipos, size=n, p=[.45,.28,.15,.08,.04])
    persona = rng.choice(personas, size=n)

    # flags de telefone (contatabilidade): tel_1 quase sempre existe, os demais menos
    fl_tel_1   = rng.binomial(1, 0.97, n)
    fl_tel_2   = rng.binomial(1, 0.55, n)
    fl_tel_3   = rng.binomial(1, 0.30, n)
    fl_tel_4   = rng.binomial(1, 0.15, n)
    fl_tel_5   = rng.binomial(1, 0.07, n)
    fl_tel_pix = rng.binomial(1, 0.60, n)

    # já contratou serviço similar (raro, mas forte preditor)
    ac_pessoais = rng.binomial(1, 0.04, n)
    fl_cap      = rng.binomial(1, 0.03, n)
    fl_hosp     = rng.binomial(1, 0.05, n)

    fl_hl = rng.binomial(1, 0.20, n)   # hotlist: cartão emitido nos últimos 3 meses

    premium = np.isin(tipo_cartao, ["PLATINUM","SIGNATURE","INFINITE"]).astype(int)
    persona_efeito = pd.Series(persona).map({
        "FAMILIA_CONSERVADORA": 0.5, "PREMIUM_VIAJANTE": 0.4, "INVESTIDOR": 0.2,
        "PRATICO": 0.0, "CONECTADO": -0.1, "JOVEM_URBANO": -0.3}).values

    # --- sinal verdadeiro (log-odds) ---------------------------------------
    lp = (0.9*fl_hl
          + 1.3*ac_pessoais + 1.1*fl_cap + 1.0*fl_hosp
          + 0.4*premium
          + 0.015*(idade - 45)
          + 0.3*fl_tel_pix
          + persona_efeito
          + rng.normal(0, 0.5, n))          # ruído irredutível

    # calibra o intercepto para bater a taxa-alvo de positivos
    lo, hi = -25.0, 10.0
    for _ in range(80):
        mid = (lo + hi) / 2
        if _sigmoid(lp + mid).mean() > taxa_alvo:
            hi = mid
        else:
            lo = mid
    b0 = (lo + hi) / 2
    prob = _sigmoid(lp + b0)
    fl_venda = rng.binomial(1, prob)

    # produto vendido só existe QUANDO houve venda (isso é leakage! ver Seção 4)
    id_prod = np.full(n, np.nan, dtype=object)
    ds_prod = np.full(n, np.nan, dtype=object)
    idx_venda = np.where(fl_venda == 1)[0]
    escolha = rng.integers(0, len(produtos), size=idx_venda.size)
    id_prod[idx_venda] = [produtos[i][0] for i in escolha]
    ds_prod[idx_venda] = [produtos[i][1] for i in escolha]

    # datas
    safra_meses = rng.integers(0, 78, n)  # até ~6,5 anos de emissão
    safra = (pd.Timestamp("2025-06-01") - pd.to_timedelta(safra_meses*30, unit="D"))
    dt_nasc = pd.Timestamp("2025-06-01") - pd.to_timedelta(idade*365, unit="D")

    # preferências: a maioria nula (como você descreveu)
    def gera_pref(p_nulo):
        v = rng.choice(prefs, size=n)
        mask = rng.random(n) < p_nulo
        v = v.astype(object); v[mask] = np.nan
        return v

    df = pd.DataFrame({
        "ano_base": 2025,
        "mes_base": 6,
        "nr_cartao": rng.integers(1_000_000_000, 9_999_999_999, n).astype(str),
        "nr_cpf": rng.integers(10_000_000_000, 99_999_999_999, n).astype(str),
        "cidade": [u + "_CID_" + str(c) for u, c in zip(uf, rng.integers(0, 400, n))],
        "uf": uf,
        "cep": rng.integers(1_000_000, 99_999_999, n).astype(str),
        "fl_tel_1": fl_tel_1, "fl_tel_2": fl_tel_2, "fl_tel_3": fl_tel_3,
        "fl_tel_4": fl_tel_4, "fl_tel_5": fl_tel_5, "fl_tel_pix": fl_tel_pix,
        "bandeira_cartao": rng.choice(bandeiras, size=n, p=[.45,.40,.12,.03]),
        "tipo_cartao": tipo_cartao,
        "sexo": rng.choice(["M","F"], size=n),
        "vencimento_fatura": rng.integers(1, 29, n),
        "safra": safra.normalize(),
        "pref_1": gera_pref(0.85),
        "pref_2": gera_pref(0.93),
        "pref_3": gera_pref(0.97),
        "ac_pessoais": ac_pessoais,
        "fl_cap": fl_cap,
        "fl_hosp": fl_hosp,
        "dt_nascimento": dt_nasc.normalize(),
        "persona": persona,
        "aa_AnoCamp": 2025,
        "mm_MesCamp": rng.integers(1, 9, n),   # 8 campanhas mensais (jan..ago)
        "fl_hl": fl_hl,
        "id_arquivo": rng.integers(1, 50, n),
        "id_campanha": rng.integers(900, 999, n),
        "id_produto_vendido": id_prod,
        "ds_produto_vendido": ds_prod,
        "fl_venda": fl_venda,
    })
    return df

df_raw = gerar_base_sintetica()
print("Shape:", df_raw.shape)
print("Taxa de conversao: {:.4%}".format(df_raw["fl_venda"].mean()))
df_raw.head()

### Como plugar a base real

Quando for rodar com os dados reais, **comente a linha `df_raw = gerar_base_sintetica()`** acima e use a célula abaixo. O contrato é simples: `df_raw` precisa ter as colunas com os mesmos nomes. O resto do notebook segue igual.


In [ ]:
# --- CARGA DA BASE REAL (descomente e ajuste o caminho) ----------------------
# df_raw = pd.read_parquet("caminho/para/sua_base.parquet")
# # ou, se for CSV muito grande, leia em chunks ou use dtypes economicos:
# # df_raw = pd.read_csv("caminho/sua_base.csv", low_memory=False)
#
# colunas_esperadas = [
#     "ano_base","mes_base","nr_cartao","nr_cpf","cidade","uf","cep",
#     "fl_tel_1","fl_tel_2","fl_tel_3","fl_tel_4","fl_tel_5","fl_tel_pix",
#     "bandeira_cartao","tipo_cartao","sexo","vencimento_fatura","safra",
#     "pref_1","pref_2","pref_3","ac_pessoais","fl_cap","fl_hosp","dt_nascimento",
#     "persona","aa_AnoCamp","mm_MesCamp","fl_hl","id_arquivo","id_campanha",
#     "id_produto_vendido","ds_produto_vendido","fl_venda",
# ]
# faltando = set(colunas_esperadas) - set(df_raw.columns)
# assert not faltando, f"Colunas faltando na base: {faltando}"
print("Usando base sintetica. Para dados reais, veja a celula acima.")

## 3. EDA e diagnóstico de desbalanceamento

Antes de modelar, olhamos três coisas que definem a estratégia: **quão raro é o positivo**, **quanto de dado falta** e **quão cara (cardinalidade) é cada coluna categórica**. Colunas com cardinalidade altíssima (como `cidade`, `cep`, `nr_cpf`) não viram feature diretamente — viram identificador, e identificador vaza ou explode a dimensionalidade.


In [ ]:
alvo = "fl_venda"
n = len(df_raw)
pos = int(df_raw[alvo].sum())
print(f"Registros........: {n:,}")
print(f"Positivos (venda): {pos:,}")
print(f"Taxa de conversao: {pos/n:.4%}")
print(f"Desbalanceamento : 1 positivo para cada {n/pos:,.0f} negativos")

In [ ]:
# Nulos por coluna (%)
nulos = (df_raw.isna().mean().sort_values(ascending=False) * 100).round(2)
print("Top colunas com nulos (%):")
print(nulos[nulos > 0])

In [ ]:
# Cardinalidade: quantos valores distintos por coluna
card = df_raw.nunique().sort_values(ascending=False)
print("Cardinalidade por coluna:")
print(card)

## 4. Vazamento de dados (*data leakage*) — o erro mais perigoso aqui

*Data leakage* é quando uma feature carrega informação que **só existe depois** de sabermos o resultado. O modelo fica "genial" no notebook e inútil na produção, porque na hora de pontuar um cliente novo aquela informação ainda não existe.

No seu schema há vazamento evidente:

- **`id_produto_vendido` e `ds_produto_vendido`** só são preenchidos *quando a venda acontece*. Usá-los como feature é dizer ao modelo "essa pessoa comprou o produto X" para prever se ela comprou. Descartar.
- **`fl_venda`** é o **alvo** — sai das features por definição.

Além disso, removemos **identificadores** (`nr_cartao`, `nr_cpf`, `id_arquivo`, `id_campanha`, `cep`, `cidade`): não têm poder preditivo genuíno, têm cardinalidade altíssima e podem memorizar clientes específicos (outra forma de vazar). Guardamos `nr_cpf` **fora das features** apenas para controlar o split (evitar o mesmo cliente em treino e teste).


In [ ]:
ALVO = "fl_venda"

# Colunas que VAZAM (só existem após a venda)
leakage_cols = ["id_produto_vendido", "ds_produto_vendido"]

# Identificadores / colunas sem valor preditivo direto
id_cols = ["nr_cartao", "nr_cpf", "id_arquivo", "id_campanha", "cep", "cidade"]

print("Removidas por leakage:", leakage_cols)
print("Removidas por serem identificadores:", id_cols)
print("Alvo:", ALVO)

## 5. Feature engineering e pré-processamento

Transformamos colunas cruas em sinal útil:

- **`tenure_meses`**: meses desde a `safra` (emissão do cartão) até a base. Cliente antigo x novo comporta-se diferente — e note que `fl_hl` (hotlist) é justamente "emitido nos últimos 3 meses", então `tenure` e `fl_hl` estão relacionados.
- **`qtd_telefones`**: soma dos `fl_tel_*`. Mais telefones = mais chance de contato efetivo (contatabilidade), que é pré-condição para vender no telemarketing.
- **`tem_pref_1`**: como `pref_*` é quase toda nula, o *fato de existir* preferência costuma valer mais que o valor em si. Viramos numa flag e descartamos `pref_2`/`pref_3`.
- **`idade`**: a base não traz idade pronta, só `dt_nascimento`. Calculamos `idade` = (data-base − `dt_nascimento`) em anos. Depois `dt_nascimento` sai das features (a informação já está em `idade`).

Depois separamos as colunas em **numéricas**, **flags** (já 0/1) e **categóricas** (que serão *one-hot*).


In [ ]:
def engenharia_features(df):
    df = df.copy()
    base = pd.to_datetime(dict(year=df["ano_base"], month=df["mes_base"], day=1))
    # idade nao vem na base: derivamos de dt_nascimento
    df["idade"] = ((base - pd.to_datetime(df["dt_nascimento"])).dt.days / 365.25).astype(int)
    df["tenure_meses"] = ((base - pd.to_datetime(df["safra"])).dt.days / 30.0).clip(lower=0)
    tel = ["fl_tel_1","fl_tel_2","fl_tel_3","fl_tel_4","fl_tel_5","fl_tel_pix"]
    df["qtd_telefones"] = df[tel].sum(axis=1)
    df["tem_pref_1"] = df["pref_1"].notna().astype(int)
    return df

df = engenharia_features(df_raw)

num_cols  = ["idade", "vencimento_fatura", "tenure_meses", "qtd_telefones", "mm_MesCamp"]
flag_cols = ["fl_tel_1","fl_tel_2","fl_tel_3","fl_tel_4","fl_tel_5","fl_tel_pix",
             "ac_pessoais","fl_cap","fl_hosp","fl_hl","tem_pref_1"]
cat_cols  = ["uf","bandeira_cartao","tipo_cartao","sexo","persona"]

features = num_cols + flag_cols + cat_cols
print(f"{len(features)} features: {len(num_cols)} num, {len(flag_cols)} flags, {len(cat_cols)} categoricas")

## 6. Split temporal — por que **não** usar split aleatório

O reflexo comum é `train_test_split` aleatório. Aqui isso seria um **erro**, por dois motivos:

1. **O futuro não pode vazar para o passado.** Em produção você treina com campanhas passadas e pontua campanhas futuras. Se embaralhar tudo, o modelo "vê o futuro" e a métrica fica otimista demais.
2. **O mesmo cliente aparece em várias campanhas/meses.** Se o mesmo `nr_cpf` cai em treino e teste, o modelo memoriza a pessoa em vez de aprender o padrão.

Então fazemos um **split temporal por período de campanha** (`aa_AnoCamp`/`mm_MesCamp`): treinamos nas campanhas mais antigas e testamos nas mais recentes. Aqui: **treino = meses 1–6, teste = meses 7–8**. Também checamos vazamento de cliente entre as janelas.


In [ ]:
periodo = df["aa_AnoCamp"] * 100 + df["mm_MesCamp"]
corte = 202506   # ate junho/2025 = treino; julho e agosto = teste

treino_mask = periodo <= corte
teste_mask  = periodo > corte

X = df[features]
y = df[ALVO].astype(int)

X_train, y_train = X[treino_mask], y[treino_mask]
X_test,  y_test  = X[teste_mask],  y[teste_mask]

print(f"Treino: {len(X_train):,} linhas | positivos: {y_train.sum():,} ({y_train.mean():.4%})")
print(f"Teste : {len(X_test):,} linhas | positivos: {y_test.sum():,} ({y_test.mean():.4%})")

# checagem de vazamento de cliente entre janelas
cpf_train = set(df.loc[treino_mask, "nr_cpf"])
cpf_test  = set(df.loc[teste_mask,  "nr_cpf"])
print(f"\nClientes em treino E teste: {len(cpf_train & cpf_test):,} "
      f"(idealmente 0; se >0, considere split por cliente)")

## 7. Pré-processador compartilhado

Construímos **um** `ColumnTransformer` usado pelos dois modelos, para a comparação ser justa:

- **numéricas**: imputação pela mediana + padronização (a Regressão Logística precisa de escala; para árvores é inofensivo).
- **flags**: passam direto (já são 0/1).
- **categóricas**: imputação + *one-hot* com `handle_unknown="ignore"` (categorias novas no teste não quebram).

Encaixotar isso num `Pipeline` garante que a transformação é **aprendida só no treino** e aplicada no teste — sem vazamento pelo pré-processamento.


In [ ]:
def make_ohe():
    # compatibilidade entre versoes do sklearn
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("flag", "passthrough", flag_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", make_ohe())]), cat_cols),
])
print("Pre-processador pronto.")

## 8. Modelo baseline — Regressão Logística

Sempre comece por um baseline simples. A **Regressão Logística** modela a probabilidade de venda como uma combinação linear das features passada por uma sigmoide. Vantagens: rápida, interpretável (cada coeficiente é um efeito direcional) e uma régua honesta — se um modelo complexo não superar isto, ele não está valendo a complexidade.

Para o desbalanceamento usamos **`class_weight="balanced"`**: internamente o modelo pesa cada positivo como se valesse ~430 negativos, forçando-o a prestar atenção na classe rara em vez de ignorá-la.


In [ ]:
logreg = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                               random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)
proba_lr = logreg.predict_proba(X_test)[:, 1]
print("Regressao Logistica treinada.")

## 9. Modelo foco — Gradient Boosting

O modelo forte é um **gradient boosting de árvores**: ele constrói centenas de arvorezinhas em sequência, cada uma corrigindo o erro da anterior. Captura interações e não-linearidades que a regressão linear não enxerga, e é o padrão-ouro para dados tabulares.

Usamos o **`HistGradientBoostingClassifier`** do próprio scikit-learn — é rápido (binning de histograma, como o LightGBM) e **não exige instalar nada além do sklearn**. Ele não tem `class_weight`, então passamos **`sample_weight`** balanceado no `fit`, com o mesmo efeito.

> Trocar por **LightGBM** ou **XGBoost** é trivial e está comentado abaixo — é aqui que você "pluga" mais modelos na abordagem foco.


In [ ]:
sw = compute_sample_weight(class_weight="balanced", y=y_train)

hgb = Pipeline([
    ("prep", preprocessor),
    ("clf", HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_depth=None,
        l2_regularization=1.0, random_state=RANDOM_STATE)),
])
hgb.fit(X_train, y_train, clf__sample_weight=sw)
proba_hgb = hgb.predict_proba(X_test)[:, 1]
print("Gradient Boosting (HistGradientBoosting) treinado.")

# --- Alternativa com LightGBM (descomente se tiver instalado) ---------------
# from lightgbm import LGBMClassifier
# n_neg, n_pos = (y_train == 0).sum(), (y_train == 1).sum()
# lgbm = Pipeline([("prep", preprocessor),
#                  ("clf", LGBMClassifier(n_estimators=500, learning_rate=0.05,
#                          scale_pos_weight=n_neg/n_pos, random_state=RANDOM_STATE))])
# lgbm.fit(X_train, y_train)
# proba_lgbm = lgbm.predict_proba(X_test)[:, 1]

## 10. Métricas certas para o problema

Aqui está o coração do raciocínio. Com 0,23% de positivos:

- **ROC-AUC** é otimista: como há muitíssimos negativos, a taxa de falso-positivo quase não se move, e o número fica alto mesmo para modelos medianos. Reportamos, mas com desconfiança.
- **PR-AUC / Average Precision** (área sob a curva *precision–recall*) é a métrica principal em base rara: foca exatamente no *trade-off* entre pegar compradores (recall) e não desperdiçar ligações (precision). O baseline "aleatório" da PR-AUC é a própria taxa de positivos (~0,0023), então qualquer valor bem acima disso já é sinal.
- **precision@k, recall@k e lift@k** são as métricas **operacionais**. Se a operação consegue ligar para os *top-k*% da base, quantos compradores caem nessa fatia? **lift@k = precision@k ÷ taxa-base** responde "quantas vezes melhor que ligar aleatoriamente". É o número que a operação entende.

`precision@k` e `lift@k` são exatamente o que importa no telemarketing ativo: você nunca liga para todo mundo, liga para o topo da lista.


In [ ]:
def precision_recall_lift_at_k(y_true, scores, frac):
    y_true = np.asarray(y_true)
    k = max(1, int(len(scores) * frac))
    topk = np.argsort(scores)[::-1][:k]
    tp = y_true[topk].sum()
    precision = tp / k
    recall = tp / y_true.sum()
    lift = precision / y_true.mean()
    return precision, recall, lift

def resumo_modelo(nome, y_true, scores):
    return {
        "modelo": nome,
        "ROC_AUC": roc_auc_score(y_true, scores),
        "PR_AUC": average_precision_score(y_true, scores),
        "prec@1%": precision_recall_lift_at_k(y_true, scores, 0.01)[0],
        "rec@1%":  precision_recall_lift_at_k(y_true, scores, 0.01)[1],
        "lift@1%": precision_recall_lift_at_k(y_true, scores, 0.01)[2],
        "rec@5%":  precision_recall_lift_at_k(y_true, scores, 0.05)[1],
        "lift@5%": precision_recall_lift_at_k(y_true, scores, 0.05)[2],
        "rec@10%": precision_recall_lift_at_k(y_true, scores, 0.10)[1],
        "lift@10%":precision_recall_lift_at_k(y_true, scores, 0.10)[2],
    }

resultados = pd.DataFrame([
    resumo_modelo("LogReg", y_test, proba_lr),
    resumo_modelo("GradBoosting", y_test, proba_hgb),
]).set_index("modelo")
print(f"Taxa-base (positivos no teste): {y_test.mean():.4%}\n")
resultados.round(4)

In [ ]:
# Curvas Precision-Recall e ROC
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for nome, sc in [("LogReg", proba_lr), ("GradBoosting", proba_hgb)]:
    p, r, _ = precision_recall_curve(y_test, sc)
    ax[0].plot(r, p, label=f"{nome} (AP={average_precision_score(y_test, sc):.3f})")
    fpr, tpr, _ = roc_curve(y_test, sc)
    ax[1].plot(fpr, tpr, label=f"{nome} (AUC={roc_auc_score(y_test, sc):.3f})")

ax[0].axhline(y_test.mean(), ls="--", color="grey", label=f"aleatorio ({y_test.mean():.4f})")
ax[0].set(xlabel="Recall", ylabel="Precision", title="Curva Precision-Recall")
ax[0].legend()
ax[1].plot([0,1],[0,1], ls="--", color="grey")
ax[1].set(xlabel="Falso Positivo", ylabel="Verdadeiro Positivo", title="Curva ROC")
ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Curva de ganho acumulado (cumulative gains): o grafico que a operacao entende
def curva_ganho(y_true, scores):
    y_true = np.asarray(y_true)
    ordem = np.argsort(scores)[::-1]
    ganho = np.cumsum(y_true[ordem]) / y_true.sum()
    eixo_x = np.arange(1, len(y_true)+1) / len(y_true)
    return eixo_x, ganho

plt.figure(figsize=(7,5))
for nome, sc in [("LogReg", proba_lr), ("GradBoosting", proba_hgb)]:
    x, g = curva_ganho(y_test, sc)
    plt.plot(x*100, g*100, label=nome)
plt.plot([0,100],[0,100], ls="--", color="grey", label="aleatorio")
plt.xlabel("% da base contatada (top-k ordenado pelo score)")
plt.ylabel("% dos compradores capturados")
plt.title("Curva de ganho acumulado")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 11. Escolha do ponto de operação — quantos ligar?

O modelo dá um *score*; a operação precisa de uma *decisão*: até que ponto da lista vale ligar. Isso é uma escolha de negócio, não estatística. A tabela abaixo mostra, para cada fatia do topo, quantos compradores você captura (`recall`) e quão eficiente é cada ligação (`precision`, `lift`). Você cruza isso com a capacidade real (nº de operadores) e o valor de uma venda para decidir o corte.


In [ ]:
fracoes = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
linhas = []
for f in fracoes:
    p, r, l = precision_recall_lift_at_k(y_test, proba_hgb, f)
    linhas.append({"top_%": f*100, "clientes_ligados": int(len(y_test)*f),
                   "precision": p, "recall": r, "lift": l})
pd.DataFrame(linhas).round(4)

## 12. A pergunta causal do hotlist

Você levantou a pergunta mais importante do projeto:

> Os clientes hotlist convertem **porque** são hotlist, ou porque os operadores **trabalham mais** neles achando que vão vender?

Isso é um problema de **confundimento (*confounding*)**. O esforço do operador é uma variável que afeta **tanto** quem é priorizado (hotlist recebe mais/melhores ligações) **quanto** o resultado (mais esforço → mais venda). Quando comparamos a conversão de hotlist vs. não-hotlist na base observacional, estamos misturando dois efeitos:

**efeito observado = efeito causal real do hotlist + efeito do esforço extra do operador**

O modelo preditivo **não resolve isso** — pior, ele *aprende e perpetua o viés*: como no histórico o hotlist converteu mais (em parte por causa do esforço), o modelo dá score alto para hotlist, a operação liga mais para eles, eles convertem, e o ciclo se retroalimenta. É uma profecia autorrealizável.

A célula abaixo **simula** esse mecanismo com a verdade conhecida, para você ver o tamanho do viés — e a seção seguinte propõe como medir o efeito real.


In [ ]:
def simula_confounding(n=200_000, seed=0):
    rng = np.random.default_rng(seed)
    hotlist = rng.binomial(1, 0.25, n)

    # Operador se esforca MAIS em quem e hotlist (esse e o confundidor)
    p_esforco = np.where(hotlist == 1, 0.70, 0.20)
    esforco = rng.binomial(1, p_esforco)

    # VERDADE que definimos: efeito causal real do hotlist e PEQUENO;
    # o efeito do esforco e GRANDE.
    EFEITO_REAL_HOTLIST = 0.30   # em log-odds
    EFEITO_ESFORCO      = 1.60   # em log-odds
    lp = -6.0 + EFEITO_REAL_HOTLIST*hotlist + EFEITO_ESFORCO*esforco
    venda = rng.binomial(1, 1/(1+np.exp(-lp)))
    return pd.DataFrame({"hotlist": hotlist, "esforco": esforco, "venda": venda})

sim = simula_confounding()

# (A) Comparacao INGENUA (o que a operacao ve hoje)
c1 = sim.loc[sim.hotlist==1, "venda"].mean()
c0 = sim.loc[sim.hotlist==0, "venda"].mean()
print("=== Comparacao ingenua (observacional) ===")
print(f"Conversao hotlist=1: {c1:.4%}")
print(f"Conversao hotlist=0: {c0:.4%}")
print(f"'Efeito' aparente do hotlist: {c1/c0:.2f}x\n")

# (B) Ajustando pelo confundidor (comparar dentro do MESMO nivel de esforco)
print("=== Ajustando pelo esforco do operador ===")
for e in [0, 1]:
    s = sim[sim.esforco==e]
    a = s.loc[s.hotlist==1,"venda"].mean()
    b = s.loc[s.hotlist==0,"venda"].mean()
    print(f"esforco={e}: hotlist=1 {a:.4%} vs hotlist=0 {b:.4%}  ->  {a/b:.2f}x")
print("\nRepare: dentro do mesmo nivel de esforco, o efeito do hotlist "
      "encolhe muito. Boa parte do 'poder' do hotlist era o esforco do operador.")

### Como medir o efeito real (e o que fazer com ele)

A simulação mostra o problema; na base real você **não conhece a verdade**, então precisa de um desenho que isole o efeito:

1. **Teste A/B (padrão-ouro).** Numa fração da base, **ignore o hotlist** na priorização e distribua o esforço aleatoriamente. Comparar conversão de hotlist vs. não-hotlist *sob esforço aleatório* dá o efeito causal limpo. É a única forma realmente confiável.
2. **Registrar o esforço.** Hoje o viés é invisível porque ninguém loga "quanto se ligou para cada cliente". Capture nº de tentativas, se houve contato, tempo de call. Sem isso, não dá para ajustar.
3. **Ajuste por confundidores / propensity score.** Com o esforço registrado, dá para estimar o efeito ajustado (como fizemos na parte B da simulação) — melhor que a comparação ingênua, mas ainda depende de ter medido os confundidores certos.
4. **Uplift modeling.** Em vez de prever "quem compra", prever "para quem a *ligação* muda a decisão". É o objetivo certo para telemarketing, mas exige dados experimentais (do item 1) para treinar.

**Recomendação prática:** trate o `fl_hl` com ceticismo. Vale treinar uma versão do modelo **sem** `fl_hl` e comparar o lift — se o modelo continua bom sem ele, você reduz a dependência de uma feature enviesada. E leve ao time de operação a proposta de um A/B com esforço aleatório numa fatia pequena: é o que transforma a sua pergunta em número.


In [ ]:
# Versao do modelo SEM fl_hl: o ranking piora muito sem essa feature?
flags_sem_hl = [c for c in flag_cols if c != "fl_hl"]
prep_sem_hl = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("flag", "passthrough", flags_sem_hl),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", make_ohe())]), cat_cols),
])
hgb_sem_hl = Pipeline([("prep", prep_sem_hl),
                       ("clf", HistGradientBoostingClassifier(
                           max_iter=300, learning_rate=0.05,
                           l2_regularization=1.0, random_state=RANDOM_STATE))])
hgb_sem_hl.fit(X_train, y_train, clf__sample_weight=sw)
proba_sem_hl = hgb_sem_hl.predict_proba(X_test)[:, 1]

comp = pd.DataFrame([
    resumo_modelo("GradBoosting c/ fl_hl", y_test, proba_hgb),
    resumo_modelo("GradBoosting s/ fl_hl", y_test, proba_sem_hl),
]).set_index("modelo")[["PR_AUC","lift@1%","lift@5%","rec@10%"]]
comp.round(4)

## 13. Explicabilidade

Para você **explicar como o algoritmo funciona** (você citou essa necessidade), usamos **importância por permutação**: embaralhamos uma feature de cada vez e medimos quanto a métrica (PR-AUC) piora. Se piorar muito, a feature era importante. É agnóstico ao modelo e mede impacto no ranking, não só no ajuste interno.


In [ ]:
# Amostra do teste para acelerar a permutacao
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(X_test), size=min(8000, len(X_test)), replace=False)
Xs, ys = X_test.iloc[idx], y_test.iloc[idx]

imp = permutation_importance(hgb, Xs, ys, n_repeats=5,
                             scoring="average_precision", random_state=RANDOM_STATE)
imp_df = (pd.DataFrame({"feature": features, "importancia": imp.importances_mean})
          .sort_values("importancia", ascending=False))

plt.figure(figsize=(7,6))
plt.barh(imp_df["feature"][::-1], imp_df["importancia"][::-1])
plt.xlabel("Queda na PR-AUC ao embaralhar (maior = mais importante)")
plt.title("Importancia por permutacao — Gradient Boosting")
plt.tight_layout(); plt.show()
imp_df.round(5)

## 14. Conclusões e próximos passos

**O que este pipeline entrega:** uma lista de clientes ordenada por propensão de compra, avaliada pelas métricas que importam no telemarketing (PR-AUC e, principalmente, precision/lift no topo da lista). O gradient boosting deve superar o baseline logístico; se não superar por margem clara, prefira o baseline pela simplicidade.

**Decisões de projeto que mais importaram:**
- Tratar o problema como **ranking**, não classificação binária com threshold 0.5.
- Métricas de **base rara** (PR-AUC, precision@k, lift) em vez de acurácia/ROC.
- **Split temporal** e cuidado com **leakage** (`id/ds_produto_vendido`).
- Balanceamento via `class_weight`/`sample_weight`, sem precisar reamostrar a base inteira.

**Próximos passos sugeridos:**
1. **Rodar na base real** trocando a célula de carga (Seção 2). Com 8,88M linhas, avalie ler em Parquet e, se precisar, treinar em subamostra dos negativos (undersampling) — o positivo é raro e deve ser mantido inteiro.
2. **Calibração de probabilidade** (`CalibratedClassifierCV`) se você quiser usar o score como probabilidade real, não só como ordem.
3. **Validação temporal em várias janelas** (treina mês a mês, testa no seguinte) para medir estabilidade.
4. **Encaminhar o A/B de esforço aleatório** (Seção 12) — é o único jeito de responder de vez à sua pergunta sobre o hotlist e de destravar *uplift modeling* no futuro.
5. **Plugar LightGBM/XGBoost** (célula comentada na Seção 9) e comparar na mesma tabela de métricas.

> Lembrete de honestidade metodológica: os números deste notebook vêm de **dados sintéticos**. Eles validam que o *pipeline* funciona; os valores reais de lift só aparecem quando você rodar na sua base.
